In [1]:
# imports
import spacy
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import defaultdict
from enum import Enum

# LLM imports
from huggingface_hub import InferenceClient
from openai import OpenAI
from urllib import response
from itertools import combinations

# load environment variables
from dotenv import load_dotenv
import os

In [2]:
# function declarations: filter

def is_relevant(sent):
    has_noun_subject = any(
        t.dep_ in ("sb", "nsubj") and t.pos_ == "NOUN" 
        for t in sent
    )
    
    has_kardinalitaet = any(
        t.text.lower() in ["jeder", "jede", "jedes", "mehrere", 
                           "verschiedene", "kein", "alle", "eine", "einen"]
        for t in sent
    )
    
    has_verb = any(t.pos_ in ("VERB", "AUX") for t in sent)
    
    noun_count = sum(1 for t in sent if t.pos_ == "NOUN")
    
    # relevant if: subject + verb + either cardinality or multiple nouns
    return has_noun_subject and has_verb and (has_kardinalitaet or noun_count >= 2)

# checks if sentence has a verb as root
def has_root_verb(sent):
    has_root_verb = None
    
    for token in sent:
        if token.pos_ in ("VERB", "AUX") and token.dep_ == "ROOT":
            has_root_verb = True
            break
    return has_root_verb



In [3]:
# function declarations rulebased extraction

# Entity management functions
def get_entity_id(noun, entities):
    """return id or create new ID"""
    global entity_id_counter
    key = noun.lemma_
    if key not in entities:
        entities[key] = {
            "id": f"e{entity_id_counter}",
            "type": noun.lemma_,
        }
        entity_id_counter += 1
    return entities[key]["id"]


def get_verb_root(token):
    """search for the main verb of the sentence"""
    # find the head of the token
    head = token.head
    while head.dep_ not in ("ROOT", "rc"):
        head = head.head
    
    # get the main verb by following auxiliary verbs
    def find_main_verb(node):
        for child in node.children:
            if child.pos_ in ("VERB", "AUX"):
                return find_main_verb(child)
        return node.i
    
    return find_main_verb(head)


def get_conjunctions(node, verb_groups, verb_idx, role):
    """recursively get all conjunctions of a noun"""
    verb_groups[verb_idx][role].append(node)
    for child in node.children:
        if child.dep_ == "cj" and child.pos_ == "NOUN":
            get_conjunctions(child, verb_groups, verb_idx, role)
        if child.dep_ == "cd" and child.pos_ == "CCONJ":
            for conj_child in child.children:
                if conj_child.pos_ == "NOUN" and conj_child.dep_ in ("nk", "da", "cj"):
                    get_conjunctions(conj_child, verb_groups, verb_idx, role)


def get_subjects(token, verb_groups, verb_idx):
    """returns all subjects of a verb"""
    if token.pos_ == "PRON":
        head = token.head
        while head.dep_ not in ("rc", "ROOT"):
            head = head.head
        if head.dep_ == "rc":
            subj = head.head
            get_conjunctions(subj, verb_groups, verb_idx, "subjects")
    else:
        get_conjunctions(token, verb_groups, verb_idx, "subjects")

def get_objects(token, verb_groups, verb_idx):
    """returns all objects of a verb"""
    get_conjunctions(token, verb_groups, verb_idx, "objects")


def get_cardinality(token):
    """returns cardinality of a noun based on determiners and morphology"""
    for child in token.children:
        if child.lemma_ in ("ein", "eine"):
            return "1"
        if child.lemma_ in ("mehrere", "viele", "alle", "einige"):
            return "n"
        if child.dep_ == "det":
            # morphology check for determiner
            number = child.morph.get("Number")
            if number == ["Plur"]:
                return "n"
            if number == ["Sing"]:
                return "1"
    # Fallback: morphology check for noun itself
    number = token.morph.get("Number")
    if number == ["Plur"]:
        return "n"
    return "1_fallback"




In [4]:
# function declarations: homonym resolution

def build_noun_table(doc):
    """builds a table of nouns with their contexts"""
    seen = {}
    for sent in doc.sents:
        for token in sent:
            if token.pos_ == "NOUN":
                lemma = token.lemma_.lower()
                
                if lemma not in seen:
                    seen[lemma] = {
                        "id": len(seen),
                        "lemma": lemma,
                        "variants": set(),
                        "contexts": [],
                        "cluster_id": None
                    }
                
                seen[lemma]["variants"].add(token.text)
                seen[lemma]["contexts"].append({
                    "sent": sent.text.strip(),
                    "token_idx": token.i
                })
    
    return seen


def get_contextual_embedding(word, sentence, tokenizer, model, max_length=512):
    """returns the average embedding of the context words (excluding the target word)"""
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=max_length)
    tokens = tokenizer.tokenize(sentence, truncation=True, max_length=max_length)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    word_tokens = tokenizer.tokenize(word)
    
    context_indices = [
        i + 1 for i, token in enumerate(tokens)
        if token not in word_tokens
    ]
    
    if not context_indices:
        return outputs.last_hidden_state[0][0].numpy()
    
    context_embeddings = outputs.last_hidden_state[0][context_indices]
    return context_embeddings.mean(dim=0).numpy()


def get_window_context(token_idx, sents, window=2):
    """returns a window of sentences around the token index"""
    sent_idx = next(
        (i for i, sent in enumerate(sents) if sent.start <= token_idx < sent.end),
        None
    )
    
    if sent_idx is None:
        return " ".join(s.text for s in sents)
    
    start = max(0, sent_idx - window)
    end = min(len(sents), sent_idx + window + 1)
    
    return " ".join(sent.text for sent in sents[start:end])


def split_homonymes(noun_table, doc, tokenizer, model, threshold=0.85):
    """splits homonymous nouns based on contextual embeddings and cosine similarity"""
    sents = list(doc.sents)
    result = {}
    
    for lemma, entry in noun_table.items():
        contexts = entry["contexts"]
        
        if len(contexts) <= 1:
            result[lemma] = entry
            continue
        
        embeddings = np.array([
            get_contextual_embedding(
                lemma,
                get_window_context(ctx["token_idx"], sents),
                tokenizer,
                model
            )
            for ctx in contexts
        ])
        
        sim_matrix = cosine_similarity(embeddings)
        sims = sim_matrix[np.triu_indices(len(embeddings), k=1)]
        
        if len(sims) == 0:
            result[lemma] = entry
            continue
        
        threshold = sims.mean() - sims.std()
        
        groups = []
        for i, emb in enumerate(embeddings):
            placed = False
            for group in groups:
                sim = cosine_similarity([emb], [embeddings[group[0]]])[0][0]
                if sim >= threshold:
                    group.append(i)
                    placed = True
                    break
            if not placed:
                groups.append([i])
        
        if len(groups) == 1:
            result[lemma] = entry
        else:
            for idx, group in enumerate(groups):
                new_key = f"{lemma}_{idx}"
                result[new_key] = {
                    "lemma": lemma,
                    "variants": entry["variants"],
                    "contexts": [contexts[i] for i in group]
                }
    
    return result


def get_noun_reference(token, sent):
    """tries to find a reference for an ambiguous noun using genitive attributes, 'von + noun' constructions, and shared subjects"""
    # 1. genitive attribute
    genitiv = [c for c in token.children if c.dep_ == "ag"]
    if genitiv:
        return genitiv[0].lemma_.lower()
    
    # 2. "von + noun"
    for t in sent:
        if t.lemma_.lower() == "von":
            for child in t.children:
                if child.pos_ == "NOUN" and child.i < token.i:
                    return child.lemma_.lower()
    
    # 3. subjects of the same verb
    verb = token.head
    if verb.dep_ == "cj":
        verb = verb.head
    if verb.dep_ == "cj":
        verb = verb.head
    
    if verb.pos_ in ("VERB", "AUX"):
        subjects = [
            t for t in sent
            if t.dep_ == "sb" and t.head == verb and t.pos_ == "NOUN" and t != token
        ]
        if subjects:
            return subjects[0].lemma_.lower()
    
    return None


def resolve_ambiguous_nouns(doc, noun_table):
    """resolves ambiguous nouns by trying to find references and creating new entries in the noun table"""
    ambiguous = {
        lemma for lemma, entry in noun_table.items()
        if len(entry["contexts"]) > 1
    }
    
    result = dict(noun_table)
    replaced_contexts = {lemma: set() for lemma in ambiguous}

    for sent in doc.sents:
        for token in sent:
            if token.pos_ != "NOUN":
                continue
            
            lemma = token.lemma_.lower()
            if lemma not in ambiguous:
                continue
            
            bezug = get_noun_reference(token, sent)
            
            if not bezug or bezug == lemma:
                continue
            if bezug not in noun_table:
                continue
            
            new_key = f"{bezug}_{lemma}"
            
            if new_key not in result:
                result[new_key] = {
                    "id": len(result),
                    "lemma": new_key,
                    "variants": {token.text},
                    "contexts": [{"sent": sent.text.strip(), "token_idx": token.i}],
                    "cluster_id": None,
                    "resolved": True
                }
            else:
                result[new_key]["variants"].add(token.text)
                result[new_key]["contexts"].append({
                    "sent": sent.text.strip(),
                    "token_idx": token.i
                })
            
            replaced_contexts[lemma].add(token.i)
    
    for lemma in ambiguous:
        if lemma not in result:
            continue
        original_contexts = [
            c for c in result[lemma]["contexts"]
            if c["token_idx"] not in replaced_contexts[lemma]
        ]
        if original_contexts:
            result[lemma]["contexts"] = original_contexts
        else:
            del result[lemma]
    
    return result


def replace_nouns(doc, noun_table):
    """replaces nouns in doc based on noun table"""
    replacements = {}
    
    for key, entry in noun_table.items():
        for context in entry["contexts"]:
            idx = context["token_idx"]
            original_form = doc[idx].text
            original_lemma = doc[idx].lemma_.lower()
            
            if "_" in key:
                prefix, base = key.rsplit("_", 1)
                flex_suffix = original_form[len(original_lemma):]
                new_label = f"{prefix.title()}_{base.title()}{flex_suffix}"
            else:
                flex_suffix = original_form[len(original_lemma):]
                new_label = key.title() + flex_suffix
            
            replacements[idx] = new_label
    
    tokens = [token.text for token in doc]
    for idx, label in replacements.items():
        tokens[idx] = label
    
    return " ".join(tokens)


In [5]:
# function declarations: synonym clustering

load_dotenv()
token = os.environ.get("HF_TOKEN")

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

HF_MODEL = "llama3.1:8b"


def cluster_synonyms_llm(noun_table, original_text):
    """clusters synonyms by LLM"""
    lemmas = [k for k in noun_table.keys() if "_" not in k]
    
    prompt = f"""Du gruppierst Wörter mit gleicher Bedeutung aus dem Text.

Hat nur ein Wort eine Bedeutung, gruppierst du es allein.
Haben mehrere Wörter die gleiche Bedeutung, gruppierst du sie zusammen.

Prüfe für jede mögliche Gruppe: Kannst du im Text das eine Wort durch das andere ersetzen ohne die Bedeutung zu ändern?
Wenn ja, dann bildest du eine Gruppe.
Wenn nein, dann bildest du keine Gruppe.

Originaltext:
{original_text}

Substantive: {lemmas}

Antworte NUR als JSON, ohne Erklärungen, ohne Markdown-Backticks:
{{
  "groups": [
    {{"label": "Bauteil", "members": ["bauteil", "teil"]}}
  ]
}}"""

    completion = client.chat.completions.create(
        model=HF_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048,
        temperature=0.1
    )
    
    text = completion.choices[0].message.content.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    start = text.find("{")
    end = text.rfind("}") + 1
    groups = json.loads(text[start:end])
    
    return groups


def apply_synonym_clusters(noun_table, clusters):
    """merged synonym-entries in noun_table"""
    result = dict(noun_table)
    
    for group in clusters["groups"]:
        label = group["label"].lower()
        members = group["members"]
        
        if len(members) <= 1:
            continue
        
        merged_contexts = []
        merged_variants = set()
        
        for member in members:
            if member in result:
                merged_contexts += result[member]["contexts"]
                merged_variants |= result[member]["variants"]
        
        result[label] = {
            "id": result[members[0]]["id"] if members[0] in result else len(result),
            "lemma": label,
            "variants": merged_variants,
            "contexts": merged_contexts,
            "cluster_id": None
        }
        
        for member in members:
            if member in result and member != label:
                del result[member]
    
    return result

In [6]:
# function declarations: komplexity reduction

def merge_relations(relations):
    """
    Checks whether two relations have the same predicate.
    If subject/object are the same or swapped:
    - Merge cardinalities using max() (n > 1)
    - Delete duplicate
    """
    def card_max(a, b):
        """n wins over 1"""
        return "n" if "n" in (a, b) else "1"

    to_delete = set()

    for i, r1 in enumerate(relations):
        if i in to_delete:
            continue

        for j, r2 in enumerate(relations):
            if j <= i or j in to_delete:
                continue

            # checks if same predicate
            if r1["predicate"] != r2["predicate"]:
                continue

            same   = r1["subject"] == r2["subject"] and r1["object"] == r2["object"]
            swapped = r1["subject"] == r2["object"]  and r1["object"] == r2["subject"]

            if same:
                # if same, merge cardinalities directly
                r1["cardinality"]["subject"] = card_max(
                    r1["cardinality"]["subject"],
                    r2["cardinality"]["subject"]
                )
                r1["cardinality"]["object"] = card_max(
                    r1["cardinality"]["object"],
                    r2["cardinality"]["object"]
                )
                to_delete.add(j)

            elif swapped:
                # if swapped, merge cardinalities crosswise
                r1["cardinality"]["subject"] = card_max(
                    r1["cardinality"]["subject"],
                    r2["cardinality"]["object"]     # crosswise merge
                )
                r1["cardinality"]["object"] = card_max(
                    r1["cardinality"]["object"],
                    r2["cardinality"]["subject"]    # crosswise merge
                )
                to_delete.add(j)

    return [r for i, r in enumerate(relations) if i not in to_delete]

# Entity to attribute
def collapse_weak_entities(entities, relations):
    """
    Removes weak entities (attribute candidates):
    - has exactly 1 relation
    - cardinality on its side is ‘1’
    - has no attributes of its own
        - is deleted, relation is deleted, its type is appended as an attribute to the other entity
    """
    to_delete_entities = set()
    to_delete_relations = set()

    # Index: entity_id → all relations + side
    from collections import defaultdict
    entity_relations = defaultdict(list)  # id → [(rel_index, side)]
    for i, rel in enumerate(relations):
        entity_relations[rel["subject"]].append((i, "subject"))
        entity_relations[rel["object"]].append((i, "object"))

    for key, entity in entities.items():
        eid = entity["id"]

        if len(entity_relations[eid]) != 1:
            continue

        rel_index, side = entity_relations[eid][0]
        rel = relations[rel_index]

        if rel["cardinality"][side] != "1":
            continue

        if entity.get("attributes"):
            continue

        other_side = "object" if side == "subject" else "subject"
        other_id = rel[other_side]

        other_key = next((k for k, e in entities.items() if e["id"] == other_id), None)
        if other_key is None:
            continue

        entities[other_key].setdefault("attributes", []).append(entity["type"])

        to_delete_entities.add(key)
        to_delete_relations.add(rel_index)

    for key in to_delete_entities:
        del entities[key]

    relations[:] = [r for i, r in enumerate(relations) if i not in to_delete_relations]

    return entities, relations

# remove duplicate annotions
def remove_duplicate_annotations(entities):
    """ if one entity has the same attribute multiple times, remove duplicates """
    for key, entity in entities.items():
        if "attributes" in entity:
            entity["attributes"] = list(set(entity["attributes"]))
    return entities

In [7]:
# variable declaration
nlp = spacy.load("de_core_news_lg")

# Limitations: Need same nouns and same verb in same relation. Change Synonyms and Homonymes.
# Perfect text example:
text1 = """Ein Kunde kann mehrere Bestellungen aufgeben.
Eine Bestellung wird von genau einem Kunden aufgegeben.
Eine Bestellung enthält mehrere Produkte.
Ein Produkt ist in mehreren Bestellungen enthalten.
Zu einem Kunden gehört genau eine Lieferadresse.
Eine Lieferadresse gehört genau einem Kunden.
Ein Produkt kann eine Beschreibung haben."""

# only nessessery information:
text2 = """Ein Kunde kann mehrere Bestellungen aufgeben.
Eine Bestellung enthält mehrere Produkte, welche in mehreren Bestellungen enthalten sein können.
Zu einem Kunden gehört genau eine Lieferadresse.
Ein Produkt kann eine Beschreibung haben."""

# phrase "in der Regel" (Redewendung)
text3 = """Ein Fluss mündet maximal in ein Meer. In ein Meer mündet mindestens ein Fluss, in der Regel aber mehrere Flüsse."""

# real exercise
text4 = """Jedes Bauteil, das verwendet wird, hat eine eindeutige Nummer, ein Fertigungsdatum und eine
Bezeichnung, die allerdings für mehrere verschiedene Bauteile gleich sein kann.
Von jedem Teil werden außerdem der Name des Herstellers, der Einkaufspreis
pro Stück und der am Lager vorhandene Vorrat gespeichert. Jedes
herzustellende Gerät hat eine eindeutige Bezeichnung. Auch von jedem schon
gefertigten Gerätetyp soll der aktuelle Lagerbestand gespeichert werden, ebenso
wie der Verkaufspreis des Gerätes. In unserem fiktiven Betrieb gilt die Regelung,
dass Maschinen, die mehr als 1000,- EUR kosten, unentgeltlich an die Kunden
ausgeliefert werden; für Geräte, die weniger kosten, ist zusätzlich zum Preis eine
gerätespezifische Anliefergebühr zu entrichten. In der Datenbank ist ebenfalls zu
speichern, welche Bauteile für welche Geräte benötigt werden. Es gibt Bauteile,
die für mehrere Geräte verwendet werden. Von jedem Kunden werden der
Name, die Adresse und die Branche gespeichert. Es kann verschiedene Kunden
mit demselben Namen oder derselben Adresse geben. Außerdem ist zu jedem
Kunden vermerkt, wer aus unserer Firma für die entsprechende
Kundenbetreuung zuständig ist. Natürlich ist auch zu speichern, welche Kunden
mit welchen Geräten beliefert werden. Es kann sein, dass gewissen Kunden für
bestimmte Geräte Sonderkonditionen eingeräumt worden sind, dies soll ggf.
ebenfalls in der Datenbank vermerkt werden."""

# real exercise reduced to relevant sentences only
text5 = """Jedes Bauteil, das verwendet wird, hat eine eindeutige Nummer, ein Fertigungsdatum und eine
Bezeichnung, die allerdings für mehrere verschiedene Bauteile gleich sein kann.
Von jedem Teil werden außerdem der Name des Herstellers, der Einkaufspreis
pro Stück und der am Lager vorhandene Vorrat gespeichert. Jedes
herzustellende Gerät hat eine eindeutige Bezeichnung. Auch von jedem schon
gefertigten Gerätetyp soll der aktuelle Lagerbestand gespeichert werden, ebenso
wie der Verkaufspreis des Gerätes. In der Datenbank ist ebenfalls zu
speichern, welche Bauteile für welche Geräte benötigt werden. Es gibt Bauteile,
die für mehrere Geräte verwendet werden. Von jedem Kunden werden der
Name, die Adresse und die Branche gespeichert. Es kann verschiedene Kunden
mit demselben Namen oder derselben Adresse geben. Außerdem ist zu jedem
Kunden vermerkt, wer aus unserer Firma für die entsprechende
Kundenbetreuung zuständig ist. Natürlich ist auch zu speichern, welche Kunden
mit welchen Geräten beliefert werden. Es kann sein, dass gewissen Kunden für
bestimmte Geräte Sonderkonditionen eingeräumt worden sind, dies soll ggf.
ebenfalls in der Datenbank vermerkt werden."""

#text = " ".join(text.split())



In [8]:
# llm model
# contextuell embedding
model = "deepset/gbert-base"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModel.from_pretrained(model)



In [9]:
# homoym resolution
# Phase 1: Rule-based
def homonym_resolution_phase1(doc, homonym_resolution):
    """resolves homonymes by rule-based heuristics"""
    
    if homonym_resolution in (homonym_resolution.RULE_BASED, homonym_resolution.FULL):
        noun_table = build_noun_table(doc)
        noun_table = resolve_ambiguous_nouns(doc, noun_table)
        text_phase1 = replace_nouns(doc, noun_table)
    else:
        text_phase1 = doc.text
    # return text_phase1
    # print("=== Nach Rule-based ===")
    # print(text_phase1)
    # with open("output.txt", "w", encoding="utf-8") as f:
    #     f.write(text_phase1)

    # Phase 2: Embedding-based
    doc2 = nlp(text_phase1)
    if homonym_resolution in (homonym_resolution.LLM_BASED, homonym_resolution.FULL):
        noun_table2 = build_noun_table(doc2)
        noun_table2 = split_homonymes(noun_table2, doc2, tokenizer, model)
        text_phase2 = replace_nouns(doc2, noun_table2)
    else:
        text_phase2 = doc2.text
        
    return text_phase2

    # print("\n=== Nach Embedding-based ===")
    # print(text_phase2)

In [10]:
# synonym clustering

def synonym_clustering(doc, text):
    """clusters synonyms by LLM"""
    # after build_noun_table and resolve_ambiguous_nouns:
    noun_table = build_noun_table(doc)
    noun_table = resolve_ambiguous_nouns(doc, noun_table)

    # synonym-clustering
    clusters = cluster_synonyms_llm(noun_table, text)
    print(json.dumps(clusters, indent=2, ensure_ascii=False))
    noun_table = apply_synonym_clusters(noun_table, clusters)
    text_phase2 = replace_nouns(doc, noun_table)
    # print(text_phase2)
    return text_phase2

In [11]:
# Main - extraction loop

class Mode(Enum):
    NONE = "none"
    RULE_BASED = "rule-based"
    LLM_BASED = "llm-based"
    FULL = "full"

# Extract entities and relations
def text2ERM(text, hr = Mode.NONE, sc = False, mr = False, cwe = False):

    entities = {}
    relations = []
    global entity_id_counter
    entity_id_counter = 1

    doc = nlp(text)
    if hr != Mode.NONE:
        text_phase1 = homonym_resolution_phase1(doc, hr)
    else:
        text_phase1 = text
    text_phase1_doc = nlp(text_phase1)

    if sc:
        text_phase2 = synonym_clustering(text_phase1_doc, text)
    else:
        text_phase2 = text_phase1_doc
    text_phase2_doc = nlp(text_phase2)

    for sent in text_phase2_doc.sents:
        
        if not has_root_verb(sent):
            continue
        # if not is_relevant(sent):
        #     continue

        # variable initialization
        verb = None
        verb_lemma = None
        subject_card = None
        object_card = None
        attribute = None
        verb_groups = defaultdict(lambda: {"subjects": [], "objects": []})

        # Identify subject and object based on dependency labels
        for token in sent:
            if token.dep_ in ("sb", "nsubj"):
                verb_idx = get_verb_root(token)
                get_subjects(token, verb_groups, verb_idx)
            elif token.dep_ in ("oa", "obj") or ((token.dep_ in ("nk", "da")) and token.pos_ == "NOUN"):
                verb_idx = get_verb_root(token)
                get_objects(token, verb_groups, verb_idx)

        # Only create a relation if we have a valid subject, verb, and object
        for verb_idx, group in verb_groups.items():
            verb_lemma = doc[verb_idx].lemma_
            if len(group["subjects"]) > 0 and len(group["objects"]) > 0:
                for subject in group["subjects"]:
                    for obj in group["objects"]:
                        subj_id = get_entity_id(subject, entities)
                        obj_id = get_entity_id(obj, entities)
                        subject_card = get_cardinality(subject)
                        object_card = get_cardinality(obj)
                        relations.append({
                            "subject": subj_id,
                            "predicate": verb_lemma,
                            "object": obj_id,
                            "cardinality": {
                                "subject": subject_card,
                                "object": object_card
                            }
                        })

    # Merge duplicate relations
    if mr:
        relations = merge_relations(relations)

    # Collapse weak entities into attributes
    if cwe:
        entities, relations = collapse_weak_entities(entities, relations)

    # Remove duplicate annotations
    remove_duplicate_annotations(entities)

    output = {
        "entities": list(entities.values()),
        "relations": relations
    }

    # print(json.dumps(output, indent=2, ensure_ascii=False))
    # with open('data.json', 'w', encoding='utf-8') as f:
    #     json.dump(output, f, indent=2, ensure_ascii=False)
    
    return output


In [12]:
# for token in doc:
#     print(token.text, token.dep_, token.pos_)

In [13]:
# import spacy
# from spacy import displacy

# text = nlp("Jedes Bauteil, das verwendet wird, hat eine eindeutige Nummer und eine Bezeichnung, die allerdings für mehrere verschiedene Bauteile gleich sein kann.")

# #displacy.serve(text, style="dep")
# svg = displacy.render(doc, style="dep", jupyter=False)

# with open("dependency.svg", "w", encoding="utf-8") as f:
#     f.write(svg)

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading
from enum import Enum as PyEnum

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
from enum import Enum

class Mode(str, PyEnum):
    NONE = "none"
    RULE_BASED = "rule-based"
    LLM_BASED = "llm-based"
    FULL = "full"

class TextInput(BaseModel):
    text: str
    homonym_resolution: Mode = Mode.NONE
    synonym_clustering: bool = False
    merge_relations: bool = False
    collapse_weak_entities: bool = False

@app.post("/extract")
def extract(body: TextInput):
    import __main__
    _text2ERM = __main__.text2ERM
    result = _text2ERM(
        body.text,
        hr=body.homonym_resolution,
        sc=body.synonym_clustering,
        mr=body.merge_relations,
        cwe=body.collapse_weak_entities,
    )
    return result

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Server läuft auf http://localhost:8000")

Server läuft auf http://localhost:8000


INFO:     Started server process [2560]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51463 - "OPTIONS /extract HTTP/1.1" 200 OK
INFO:     127.0.0.1:51463 - "POST /extract HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 410, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\fastapi\applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\starlette\middlewar

INFO:     127.0.0.1:51464 - "OPTIONS /extract HTTP/1.1" 200 OK
INFO:     127.0.0.1:51464 - "POST /extract HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 410, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\fastapi\applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\starlette\middlewar

INFO:     127.0.0.1:51465 - "OPTIONS /extract HTTP/1.1" 200 OK
INFO:     127.0.0.1:51465 - "POST /extract HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 410, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\fastapi\applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\Arbeit\Documents\Uni\NLP\Projekt\.venv\Lib\site-packages\starlette\middlewar